# Evaluating Multiple LM Outputs (External)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# imports
import json
import os
import pandas as pd
import importlib.util
import sys
from os.path import join
from copy import deepcopy
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion
from stat_genie.blade_pipeline.additions.analysis.fix_code import \
    check_and_fix_code

In [3]:
# load files
analysis_subdir_path_1 = "outputs/analysis1_output"
analysis_subdir_path_2 = "outputs/analysis2_output"
analysis_subdir_path_3 = "outputs/analysis3_output"

multirun_filename_1 = "multirun_analyses.json"
multirun_filename_2 = "multirun_analyses.json"
multirun_filename_3 = "multirun_analyses.json"

# use both files to get analysis code paths
multirun_path_1 = join(analysis_subdir_path_1, multirun_filename_1)
multirun_path_2 = join(analysis_subdir_path_2, multirun_filename_2)
multirun_path_3 = join(analysis_subdir_path_3, multirun_filename_3)

with open(multirun_path_1, "r") as file:
    multirun_analyses_1 = json.load(file)

with open(multirun_path_2, "r") as file:
    multirun_analyses_2 = json.load(file)
    
with open(multirun_path_3, "r") as file:
    multirun_analyses_3 = json.load(file)

num_analyses_1 = multirun_analyses_1['n']
num_analyses_2 = multirun_analyses_2['n']
num_analyses_3 = multirun_analyses_3['n']

analysis_code_filenames_1 = [f"llm_analysis_{i}.py" for i in range(num_analyses_1)]
analysis_code_filenames_2 = [f"llm_analysis_{i}.py" for i in range(num_analyses_2)]
analysis_code_filenames_3 = [f"llm_analysis_{i}.py" for i in range(num_analyses_3)]

analysis_code_paths_1 = [join(analysis_subdir_path_1, filename)
                         for filename in analysis_code_filenames_1]

analysis_code_paths_2 = [join(analysis_subdir_path_2, filename)
                         for filename in analysis_code_filenames_2]

analysis_code_paths_3 = [join(analysis_subdir_path_3, filename)
                         for filename in analysis_code_filenames_3]

In [4]:
llm_provider = "openai"
llm_model = "gpt-5-mini"
llm_assistant = llm(provider=llm_provider, model=llm_model)

[2025-12-05 11:20:17.86][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/campus/austin.zane/stat-genie/config/llm_config.yml'.


In [5]:
features_1 = format_features(multirun_analyses_1, num_analyses_1, llm_assistant)
features_2 = format_features(multirun_analyses_2, num_analyses_2, llm_assistant)
features_3 = format_features(multirun_analyses_3, num_analyses_3, llm_assistant)

[2025-12-05 11:20:18.35][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:20:27.80][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  9.45 seconds
[2025-12-05 11:20:27.80][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:20:27.82][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:20:51.47][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  23.65 seconds
[2025-12-05 11:20:51.47][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:20:51.48][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:20:57.26][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.78

[2025-12-05 07:59:00.40][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  11.20 seconds
[2025-12-05 07:59:00.40][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 07:59:00.41][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 07:59:08.24][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.83 seconds
[2025-12-05 07:59:08.24][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 07:59:08.25][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 07:59:15.34][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.09 seconds
[2025-12-05 07:59:15.34][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[

In [6]:
model_info_1 = format_model_info(multirun_analyses_1, num_analyses_1, llm_assistant)
model_info_2 = format_model_info(multirun_analyses_2, num_analyses_2, llm_assistant)
model_info_3 = format_model_info(multirun_analyses_3, num_analyses_3, llm_assistant)

[2025-12-05 11:30:45.48][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:31:04.35][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  18.88 seconds
[2025-12-05 11:31:04.35][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:31:04.36][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:31:22.47][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  18.11 seconds
[2025-12-05 11:31:22.47][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:31:22.50][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:31:36.41][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  13.

In [7]:
# load dataset, need more user-friendly input method later
dataset_name = multirun_analyses_1['dataset_name']
dataset_path = join("..", "..", "..", "src", "stat_genie", "blade_pipeline",
                    "datasets", dataset_name, "data.csv")
absolute_dataset_path = os.path.abspath(dataset_path)
data = pd.read_csv(dataset_path)

In [8]:
# NOTE: Code fixing for llm_analysis_*.py files is now handled during generation
# in the analysis notebooks (with fix_code=True in MultiRunConfig).
# This cell has been removed to avoid redundant code fixing.

In [9]:
transform_functions_1 = {}
transform_functions_2 = {}
transform_functions_3 = {}
model_functions_1 = {}
model_functions_2 = {}
model_functions_3 = {}

# ----- get the transform and model functions for the first input group -----
for i, analysis_code_path in enumerate(analysis_code_paths_1):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_1_{i}"] = module
    spec.loader.exec_module(module)
    
    transform_functions_1[i] = module.transform
    model_functions_1[i] = module.model

# ----- get the transform and model functions for the second input group -----
for i, analysis_code_path in enumerate(analysis_code_paths_2):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_2_{i}"] = module
    spec.loader.exec_module(module)

    transform_functions_2[i] = module.transform
    model_functions_2[i] = module.model

# ----- get the transform and model functions for the third input group -----
for i, analysis_code_path in enumerate(analysis_code_paths_3):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_3_{i}"] = module
    spec.loader.exec_module(module)

    transform_functions_3[i] = module.transform
    model_functions_3[i] = module.model

In [10]:
transformed_datasets_1 = {}
for i, transform_func in transform_functions_1.items():
    try:
        transformed_datasets_1[i] = transform_func(data.copy())
        print(f"[Transform 1-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform 1-{i}] Failed with error: {e}")
        transformed_datasets_1[i] = None

transformed_datasets_2 = {}
for i, transform_func in transform_functions_2.items():
    try:
        transformed_datasets_2[i] = transform_func(data.copy())
        print(f"[Transform 2-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform 2-{i}] Failed with error: {e}")
        transformed_datasets_2[i] = None
        
transformed_datasets_3 = {}
for i, transform_func in transform_functions_3.items():
    try:
        transformed_datasets_3[i] = transform_func(data.copy())
        print(f"[Transform 3-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform 3-{i}] Failed with error: {e}")
        transformed_datasets_3[i] = None


model_results_1 = {}
for i, model_func in model_functions_1.items():
    try:
        if transformed_datasets_1[i] is None:
            print(f"[Model 1-{i}] Skipping — transform failed.")
            continue

        model_results_1[i] = model_func(transformed_datasets_1[i].copy())
        print(f"[Model 1-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Model 1-{i}] Failed with error: {e}")
        model_results_1[i] = None

model_results_2 = {}
for i, model_func in model_functions_2.items():
    try:
        if transformed_datasets_2[i] is None:
            print(f"[Model 2-{i}] Skipping — transform failed.")
            continue

        model_results_2[i] = model_func(transformed_datasets_2[i].copy())
        print(f"[Model 2-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Model 2-{i}] Failed with error: {e}")
        model_results_2[i] = None
        
model_results_3 = {}
for i, model_func in model_functions_3.items():
    try:
        if transformed_datasets_3[i] is None:
            print(f"[Model 3-{i}] Skipping — transform failed.")
            continue

        model_results_3[i] = model_func(transformed_datasets_3[i].copy())
        print(f"[Model 3-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Model 3-{i}] Failed with error: {e}")
        model_results_3[i] = None

[Transform 1-0] Completed successfully.
[Transform 1-1] Completed successfully.
[Transform 1-2] Completed successfully.
[Transform 2-0] Completed successfully.
[Transform 2-1] Completed successfully.
[Transform 2-2] Completed successfully.
[Transform 3-0] Completed successfully.
[Transform 3-1] Completed successfully.
[Transform 3-2] Completed successfully.
[Model 1-0] Completed successfully.
[Model 1-1] Completed successfully.
[Model 1-2] Completed successfully.
[Model 2-0] Completed successfully.
[Model 2-1] Completed successfully.
[Model 2-2] Completed successfully.
OLS on log(fish_per_hour) summary:
                            OLS Regression Results                            
Dep. Variable:      log_fish_per_hour   R-squared:                       0.507
Model:                            OLS   Adj. R-squared:                  0.499
Method:                 Least Squares   F-statistic:                     72.48
Date:                Fri, 05 Dec 2025   Prob (F-statistic):           1.9

/accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:445: RuntimeWarning: invalid 

In [11]:
info_json_path = join("..", "..", "..", "src", "stat_genie", "blade_pipeline",
                      "datasets", dataset_name, "info.json")
with open(info_json_path, "r") as file:
    info_json = json.load(file)

task = info_json['research_questions']

for i in range(num_analyses_1):

    independent_variable = features_1[i]['independent_variables']
    dependent_variable = features_1[i]['response_variables']

    model_code = multirun_analyses_1['analyses'][str(i)]['m_code']
    model_output = model_results_1[i]

    write_final_answer_code(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        os.path.abspath(analysis_subdir_path_1),
        i,
        model_output
    )

for i in range(num_analyses_2):

    independent_variable = features_2[i]['independent_variables']
    dependent_variable = features_2[i]['response_variables']

    model_code = multirun_analyses_2['analyses'][str(i)]['m_code']
    model_output = model_results_2[i]

    write_final_answer_code(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        os.path.abspath(analysis_subdir_path_2),
        i,
        model_output
    )

for i in range(num_analyses_3):

    independent_variable = features_3[i]['independent_variables']
    dependent_variable = features_3[i]['response_variables']

    model_code = multirun_analyses_3['analyses'][str(i)]['m_code']
    model_output = model_results_3[i]

    write_final_answer_code(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        os.path.abspath(analysis_subdir_path_3),
        i,
        model_output
    )


[2025-12-05 11:33:02.63][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:34:00.29][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  57.66 seconds
[2025-12-05 11:34:00.29][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:34:00.32][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:34:46.40][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  46.08 seconds
[2025-12-05 11:34:46.40][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:34:46.41][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:35:30.12][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  43.

[2025-12-05 08:21:09.76][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  43.80 seconds
[2025-12-05 08:21:09.76][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 08:21:09.78][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 08:21:53.48][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  43.70 seconds
[2025-12-05 08:21:53.48][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 08:21:53.49][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 08:22:52.66][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  59.17 seconds
[2025-12-05 08:22:52.66][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)

In [12]:
answer_code_paths_1 = [join(analysis_subdir_path_1, f"llm_answer_{i}.py") for i in range(num_analyses_1)]
answer_code_paths_2 = [join(analysis_subdir_path_2, f"llm_answer_{i}.py") for i in range(num_analyses_2)]
answer_code_paths_3 = [join(analysis_subdir_path_3, f"llm_answer_{i}.py") for i in range(num_analyses_3)]

In [13]:
# check that code works
answer_code_paths = [answer_code_paths_1, answer_code_paths_2,
                     answer_code_paths_3]
model_results = [model_results_1, model_results_2,
                 model_results_3]
for i in range(len(answer_code_paths)):
    for j, answer_code_path in enumerate(answer_code_paths[i]):
        # get absolute path using relative path so that the helper function works correctly
        absolute_path = os.path.abspath(answer_code_path)
        # call helper function to ensure code correctness
        num_iterations = check_and_fix_code(f"llm_answer_{j}",
                                            absolute_path,
                                            "final_answer",
                                            llm_provider,
                                            llm_model,
                                            model_output=model_results[i][j],
                                            verbose=False)
        print(f"Answer {i} iteration {j} required {num_iterations} correction iterations.")

[2025-12-05 11:38:19.76][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/campus/austin.zane/stat-genie/config/llm_config.yml'.
[2025-12-05 11:38:19.88][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:39:09.20][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  49.32 seconds
[2025-12-05 11:39:09.20][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
Answer 0 iteration 0 required 1 correction iterations.
Answer 0 iteration 1 required 0 correction iterations.
Answer 0 iteration 2 required 0 correction iterations.
Answer 1 iteration 0 required 0 correction iterations.
Answer 1 iteration 1 required 0 correction iterations.
Answer 1 iteration 2 required 0 correction iterations.
Answer 2 iteration 0 required 0 correction iterations.
Answer 2 iteration 1 required 0 correction iterations.
Answer

/accounts/campus/austin.zane/stat-genie/examples/feature_perturbation/fish/outputs/analysis2_output/llm_answer_1.py:73: RuntimeWarning: overflow encountered in exp
  rr_ci_high = float(np.exp(ci_high)) if ci_high is not None else None


[2025-12-05 08:26:46.14][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 08:27:36.25][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  50.11 seconds
[2025-12-05 08:27:36.25][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
Answer 1 iteration 2 required 1 correction iterations.
Answer 2 iteration 0 required 0 correction iterations.
Answer 2 iteration 1 required 0 correction iterations.
Answer 2 iteration 2 required 0 correction iterations.


In [14]:
# get final answer functions in dict
final_answer_functions_1 = {}
final_answer_functions_2 = {}
final_answer_functions_3 = {}

# ----- get the final answer functions for the first input group -----
for i, answer_code_path in enumerate(answer_code_paths_1):
    spec = importlib.util.spec_from_file_location(f"llm_answer_{i}",
                                                  answer_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_answer_1_{i}"] = module
    spec.loader.exec_module(module)
    
    final_answer_functions_1[i] = module.extract_final_answer

# ----- get the final answer functions for the second input group -----
for i, answer_code_path in enumerate(answer_code_paths_2):
    spec = importlib.util.spec_from_file_location(f"llm_answer_{i}",
                                                  answer_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_answer_2_{i}"] = module
    spec.loader.exec_module(module)

    final_answer_functions_2[i] = module.extract_final_answer
    
# ----- get the final answer functions for the third input group -----
for i, answer_code_path in enumerate(answer_code_paths_3):
    spec = importlib.util.spec_from_file_location(f"llm_answer_{i}",
                                                  answer_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_answer_3_{i}"] = module
    spec.loader.exec_module(module)

    final_answer_functions_3[i] = module.extract_final_answer


In [15]:
final_answers_1 = {}
for i, final_answer_func in final_answer_functions_1.items():
    try:
        model_output = deepcopy(model_results_1[i])
        final_answers_1[i] = final_answer_func(model_output)
        print(f"[Answer 1-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Answer 1-{i}] Failed with error: {e}")
        final_answers_1[i] = None

final_answers_2 = {}
for i, final_answer_func in final_answer_functions_2.items():
    try:
        model_output = deepcopy(model_results_2[i])
        final_answers_2[i] = final_answer_func(model_output)
        print(f"[Answer 2-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Answer 2-{i}] Failed with error: {e}")
        final_answers_2[i] = None
        
final_answers_3 = {}
for i, final_answer_func in final_answer_functions_3.items():
    try:
        model_output = deepcopy(model_results_3[i])
        final_answers_3[i] = final_answer_func(model_output)
        print(f"[Answer 3-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Answer 3-{i}] Failed with error: {e}")
        final_answers_3[i] = None

[Answer 1-0] Completed successfully.
[Answer 1-1] Completed successfully.
[Answer 1-2] Completed successfully.
[Answer 2-0] Completed successfully.
[Answer 2-1] Completed successfully.
[Answer 2-2] Completed successfully.
[Answer 3-0] Completed successfully.
[Answer 3-1] Completed successfully.
[Answer 3-2] Completed successfully.


/accounts/campus/austin.zane/stat-genie/examples/feature_perturbation/fish/outputs/analysis2_output/llm_answer_1.py:73: RuntimeWarning: overflow encountered in exp
  rr_ci_high = float(np.exp(ci_high)) if ci_high is not None else None


In [16]:
conclusions_1 = {}

for i in range(num_analyses_1):
    independent_variable = features_1[i]['independent_variables']
    dependent_variable = features_1[i]['response_variables']

    model_code = multirun_analyses_1['analyses'][str(i)]['m_code']
    # read in final answer code from file "llm_answer_{i}.py" at analysis_subdir_path_1
    code_filename = f"llm_answer_{i}.py"
    code_path = os.path.abspath(os.path.join(analysis_subdir_path_1, code_filename))
    with open(code_path, "r", encoding="utf-8") as f:
        interpretation_code_str = f.read()
    interpretation_output = final_answers_1[i]

    conclusions_1[i] = make_conclusion(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        interpretation_code_str,
        interpretation_output
    )


conclusions_2 = {}

for i in range(num_analyses_2):
    independent_variable = features_2[i]['independent_variables']
    dependent_variable = features_2[i]['response_variables']

    model_code = multirun_analyses_2['analyses'][str(i)]['m_code']

    # read in final answer code from file "llm_answer_{i}.py" at analysis_subdir_path_2
    code_filename = f"llm_answer_{i}.py"
    code_path = os.path.abspath(os.path.join(analysis_subdir_path_2, code_filename))
    with open(code_path, "r", encoding="utf-8") as f:
        interpretation_code_str = f.read()
    interpretation_output = final_answers_2[i] if i < len(final_answers_2) else None

    conclusions_2[i] = make_conclusion(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        interpretation_code_str,
        interpretation_output
    )
    
conclusions_3 = {}

for i in range(num_analyses_3):
    independent_variable = features_3[i]['independent_variables']
    dependent_variable = features_3[i]['response_variables']

    model_code = multirun_analyses_3['analyses'][str(i)]['m_code']

    # read in final answer code from file "llm_answer_{i}.py" at analysis_subdir_path_3
    code_filename = f"llm_answer_{i}.py"
    code_path = os.path.abspath(os.path.join(analysis_subdir_path_3, code_filename))
    with open(code_path, "r", encoding="utf-8") as f:
        interpretation_code_str = f.read()
    interpretation_output = final_answers_3[i] if i < len(final_answers_3) else None

    conclusions_3[i] = make_conclusion(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        interpretation_code_str,
        interpretation_output
    )

[2025-12-05 11:39:10.17][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:39:17.22][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.05 seconds
[2025-12-05 11:39:17.22][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:39:17.24][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:39:24.95][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.71 seconds
[2025-12-05 11:39:24.95][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:39:24.97][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:39:32.55][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.58 

[2025-12-05 08:27:42.07][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.62 seconds
[2025-12-05 08:27:42.07][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 08:27:42.08][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 08:27:48.85][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.76 seconds
[2025-12-05 08:27:48.85][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 08:27:48.86][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 08:27:51.76][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  2.91 seconds
[2025-12-05 08:27:51.76][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2

In [17]:
llm_judge = llm(provider=llm_provider, model=llm_model)
data_head = data.head(10)

[2025-12-05 11:40:16.53][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/campus/austin.zane/stat-genie/config/llm_config.yml'.


In [18]:
judge_system_prompt = (
    "You are a meticulous research design evaluator. "
    "Your role is to compare two experimental trials methodologically **and interpretively**.\n\n"
    "You will go through the following reasoning plan step-by-step (internally):\n"
    "1. Understand the research question and dataset context.\n"
    "2. Examine independent, control, and response variables for both trials.\n"
    "3. Analyze the model specifications for structural or methodological similarity.\n"
    "4. Focus more on the content, less on the format.\n"
    "5. Assess whether the trials' conclusions are logically consistent given their setups.\n"
    "6. Detect whether either input is None, invalid, erroneous, or incomplete.\n"
    "   - If **one trial** shows errors or missing components but the other is valid, "
    "     impose a **strong penalty** (reduce all category scores by at least 1 point, "
    "     and cap overall similarity at 2).\n"
    "7. Synthesize your evaluation across all components.\n"
    "8. Output a numerical rating for each category.\n\n"
    "DO NOT include your reasoning — only the final dictionary.\n\n"
    "Scoring scale:\n"
    "1 = completely different\n"
    "2 = somewhat different\n"
    "3 = moderately similar\n"
    "4 = very similar\n"
    "5 = almost identical\n\n"
    "Return output **strictly in dictionary format**:\n"
    "{\n"
    "  \"independent_variables\": <number>,\n"
    "  \"control_variables\": <number>,\n"
    "  \"response_variables\": <number>,\n"
    "  \"model_specification\": <number>,\n"
    "  \"conclusions\": <number>,\n"
    "  \"overall_similarity\": <number>\n"
    "}"
)


def make_judge_prompt(task, data_head, featA, featB, modelA, modelB, conclA, conclB):
    return (
        f"Research Question / Context:\n{task}\n\n"
        "Here is a sample of the dataset to understand the structure and variables:\n"
        f"{data_head}\n\n"
        "Compare the two trials methodologically and interpretively based on the provided variables, model specifications, and conclusions.\n\n"
        "==================== TRIAL A ====================\n\n"
        "Independent Variables:\n"
        f"{featA['independent_variables']}\n\n"
        "Control Variables:\n"
        f"{featA.get('control_variables')}\n\n"
        "Response Variables:\n"
        f"{featA['response_variables']}\n\n"
        "Model Specification:\n"
        f"{modelA}\n\n"
        "Conclusion:\n"
        f"{conclA}\n\n"
        "==================== TRIAL B ====================\n\n"
        "Independent Variables:\n"
        f"{featB['independent_variables']}\n\n"
        "Control Variables:\n"
        f"{featB.get('control_variables')}\n\n"
        "Response Variables:\n"
        f"{featB['response_variables']}\n\n"
        "Model Specification:\n"
        f"{modelB}\n\n"
        "Conclusion:\n"
        f"{conclB}\n\n"
        "Now, following your reasoning plan, provide similarity ratings as JSON only."
    )


In [19]:
### judge results within each group and between each group.
# within each group there should be 3 choose 2 = 3 pairwise comparisons
# between each group there should be 3 x 3 = 9 pairwise comparisons

In [ ]:
### begin with within-group performance
within_group = {1: {}, 2: {}, 3: {}}
for i in range(num_analyses_1):
    for j in range(i + 1, num_analyses_1):
        prompt = make_judge_prompt(
            task,
            data_head,
            features_1[i],
            features_1[j],
            multirun_analyses_1['analyses'][str(i)]['m_code'],
            multirun_analyses_1['analyses'][str(j)]['m_code'],
            conclusions_1[i],
            conclusions_1[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        within_group[1][(i, j)] = response_dict
for i in range(num_analyses_2):
    for j in range(i + 1, num_analyses_2):
        prompt = make_judge_prompt(
            task,
            data_head,
            features_2[i],
            features_2[j],
            multirun_analyses_2['analyses'][str(i)]['m_code'],
            multirun_analyses_2['analyses'][str(j)]['m_code'],
            conclusions_2[i],
            conclusions_2[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        within_group[2][(i, j)] = response_dict
for i in range(num_analyses_3):
    for j in range(i + 1, num_analyses_3):
        prompt = make_judge_prompt(
            task,
            data_head,
            features_3[i],
            features_3[j],
            multirun_analyses_3['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_3[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        within_group[3][(i, j)] = response_dict

[2025-12-05 11:40:16.72][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:40:26.57][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  9.84 seconds
[2025-12-05 11:40:26.57][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:40:26.59][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:40:37.34][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  10.75 seconds
[2025-12-05 11:40:37.34][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 11:40:37.37][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 11:40:44.92][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.55

[2025-12-05 08:28:48.00][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  11.03 seconds
[2025-12-05 08:28:48.00][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 08:28:48.01][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 08:29:02.50][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  14.48 seconds
[2025-12-05 08:29:02.50][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 08:29:02.51][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 08:29:15.87][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  13.36 seconds
[2025-12-05 08:29:15.87][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)

In [ ]:
### now do between-group performance
between_group = { (1, 2): {}, (1, 3): {}, (2, 3): {} }
for i in range(num_analyses_1):
    for j in range(num_analyses_2):
        prompt = make_judge_prompt(
            task,
            data_head,
            features_1[i],
            features_2[j],
            multirun_analyses_1['analyses'][str(i)]['m_code'],
            multirun_analyses_2['analyses'][str(j)]['m_code'],
            conclusions_1[i],
            conclusions_2[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(1, 2)][(i, j)] = response_dict
for i in range(num_analyses_1):
    for j in range(num_analyses_3):
        prompt = make_judge_prompt(
            task,
            data_head,
            features_1[i],
            features_3[j],
            multirun_analyses_1['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_1[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(1, 3)][(i, j)] = response_dict
for i in range(num_analyses_2):
    for j in range(num_analyses_3):
        prompt = make_judge_prompt(
            task,
            data_head,
            features_2[i],
            features_3[j],
            multirun_analyses_2['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_2[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(2, 3)][(i, j)] = response_dict

In [ ]:
within_group

In [ ]:
between_group

In [ ]:
# get average similarity score for each subcategory within each group
average_within_group = {}
for group_id, comparisons in within_group.items():
    category_sums = {
        "independent_variables": 0,
        "control_variables": 0,
        "response_variables": 0,
        "model_specification": 0,
        "conclusions": 0,
        "overall_similarity": 0
    }
    num_comparisons = len(comparisons)
    
    for comparison, scores in comparisons.items():
        for category, score in scores.items():
            category_sums[category] += score
    
    average_scores = {category: total / num_comparisons
                      for category, total in category_sums.items()}
    average_within_group[group_id] = average_scores

In [ ]:
# show average within group rounded to nearest tenth
average_within_group

In [ ]:
# get average similarity score for each subcategory between each group
average_between_group = {}
for group_pair, comparisons in between_group.items():
    category_sums = {
        "independent_variables": 0,
        "control_variables": 0,
        "response_variables": 0,
        "model_specification": 0,
        "conclusions": 0,
        "overall_similarity": 0
    }
    num_comparisons = len(comparisons)
    
    for comparison, scores in comparisons.items():
        for category, score in scores.items():
            category_sums[category] += score
    
    average_scores = {category: total / num_comparisons
                      for category, total in category_sums.items()}
    average_between_group[group_pair] = average_scores

In [ ]:
average_between_group

In [ ]:
# summary of results
print("=" * 80)
print("pairwise similarity results")
print("=" * 80)

print("\nwithin-group consistency")
print("-" * 80)
for group_id, scores in average_within_group.items():
    group_name = {1: "baseline (no perturbation)", 2: "anonymized names", 3: "shuffled names"}[group_id]
    overall = scores['overall_similarity']
    status = "good" if overall >= 3.5 else "moderate" if overall >= 3.0 else "low"
    print(f"\ngroup {group_id} ({group_name}):")
    print(f"  overall similarity: {overall:.2f} ({status})")
    print(f"  - independent variables: {scores['independent_variables']:.2f}")
    print(f"  - control variables: {scores['control_variables']:.2f}")
    print(f"  - response variables: {scores['response_variables']:.2f}")
    print(f"  - model specification: {scores['model_specification']:.2f}")
    print(f"  - conclusions: {scores['conclusions']:.2f}")

print("\n\nbetween-group comparisons")
print("-" * 80)
comparison_names = {
    (1, 2): "baseline vs anonymized",
    (1, 3): "baseline vs shuffled",
    (2, 3): "anonymized vs shuffled"
}
for pair, scores in average_between_group.items():
    overall = scores['overall_similarity']
    baseline_score = average_within_group[1]['overall_similarity']
    drop = baseline_score - overall if pair[0] == 1 else None
    
    if pair == (1, 2):
        status = "robust" if overall >= 3.8 else "some degradation" if overall >= 3.0 else "poor"
        print(f"\n{pair} ({comparison_names[pair]}):")
        print(f"  overall similarity: {overall:.2f} ({status})")
        print(f"  tests whether the model can work without semantic name cues")
    elif pair == (1, 3):
        status = "robust" if overall >= 3.5 else "affected" if overall >= 3.0 else "heavily affected"
        print(f"\n{pair} ({comparison_names[pair]}):")
        print(f"  overall similarity: {overall:.2f} ({status})")
        if drop:
            print(f"  drop from baseline: {drop:.2f} points")
        print(f"  tests whether the model is fooled by misleading names")
    else:
        status = "similar" if overall >= 3.0 else "different"
        print(f"\n{pair} ({comparison_names[pair]}):")
        print(f"  overall similarity: {overall:.2f} ({status})")
    
    print(f"  - independent variables: {scores['independent_variables']:.2f}")
    print(f"  - control variables: {scores['control_variables']:.2f}")
    print(f"  - response variables: {scores['response_variables']:.2f}")
    print(f"  - model specification: {scores['model_specification']:.2f}")
    print(f"  - conclusions: {scores['conclusions']:.2f}")

print("\n\nnotes")
print("-" * 80)
baseline_consistency = average_within_group[1]['overall_similarity']
anon_vs_baseline = average_between_group[(1, 2)]['overall_similarity']
shuffled_vs_baseline = average_between_group[(1, 3)]['overall_similarity']
shuffled_consistency = average_within_group[3]['overall_similarity']

notes = []
if baseline_consistency < 3.0:
    notes.append("high variability even in baseline condition")
if anon_vs_baseline >= 3.8:
    notes.append("model is robust to anonymization - works well without semantic names")
elif anon_vs_baseline < 3.0:
    notes.append("model struggles without semantic name cues")
if shuffled_vs_baseline < 3.0:
    notes.append("model is heavily affected by shuffled/misleading names")
    if shuffled_consistency < 3.0:
        notes.append("  shuffled condition also shows low internal consistency")
elif shuffled_vs_baseline >= 3.5:
    notes.append("model resists misleading names effectively")

if average_between_group[(1, 3)]['conclusions'] < average_between_group[(1, 3)]['model_specification']:
    notes.append("conclusions more affected than model specifications - interpretation instability")

if not notes:
    notes.append("overall: model shows reasonable robustness to feature name perturbations")

for i, note in enumerate(notes, 1):
    print(f"{i}. {note}")

print("\n" + "=" * 80)
